# 正则化


正则化的分类有：L1正则化：权重可以变为0，相当于降维、L2正则化：权重可以无限趋近于0、dropout（随机失活）、BN(批量归一化)

In [1]:
import torch
import torch.nn as nn

In [18]:
def dropout():
    """
    dropout正则化是指在训练时，每批次中有概率为p的部分神经元随机停止参与计算（死亡），剩下的神经元按照1/(1-p)进行缩放，防止过拟合
    在激活函数后设置
    :return: none
    """
    # 1.创建隐藏层输出结果
    t1 = torch.randint(0, 10, size=(1, 4)).float()    # 创建1行4列的数据，并将数据转换为浮点型
    # print(t1)
    # 2.创建下一层进行加权求和 和 激活函数计算
    # 2.1 创建全连接层（充当线性层）
    linear1 = nn.Linear(4, 5)    # (输入神经元/输入特征维度，输出神经元/输出特征维度)
    # 2.2 加权求和 与 激活函数计算
    l1 = linear1(t1)    # 加权求和 y = xA^T + b (x=t1, A^T按照nn.Linear(a,b)自动生成能做运算的矩阵)
    print(f'加权求和后为:{l1}')
    output = torch.relu(l1)
    print(f'relu处理后为:{output}')    # 注意！！这里如果产生0值并非随机失活，而是relu处理
                                      # relu的结果是max(0,x) -> 如果比0小，则处理为0，否则为源值

    # 3.进行随机失活处理
    dropOut = nn.Dropout(p=0.5) # 每个神经元都有50%的概率失活
    d1 = dropOut(output)        # d1为随机失活后的数据，未失活的按照1/(1-p)进行缩放
    print(f'随机失活后的数据为:{d1}')


In [21]:
def BN_2n():
    """
    BN(batch normalization)叫做批量归一化，先对数据做标准化，再对数据进行重构（缩放+平移），可以减少内部协方差偏移。内部协方差：每一层的调整对后一层都有影响。
    为什么要重构？因为标准化会导致数据丢失一些信息，通过重构找回来。
    y = lambda(w缩放) * 标准化后的数据(x) + beta(b平移) 标准化采用zscore并加上小参数；w和b都是模型自动的
    :return:none
    """
    """
    BatchNorm1d:用于处理一维数据活全连接层，例如文本处理，接受形状为(N, num_features)的张量作为输入
    BatchNorm2d:主要用于卷积神经网络，处理二维图像数据或特征图。接受形状为(N, C, H, W)的张量作为输入
    BatchNorm3d:主要用于三维卷积神经网络，处理三维数据，如视频、医学图像，接收(N, C, D, H, W)的张量作为输入
    """
    # 1.创建图像样本数据
    input_2d = torch.randn(size=(1, 2, 3, 4)) # 1张图片，2个通道，3行4列(像素点)
    # 2.创建批量归一化(BN层)
                          # 输入特征数 = 通道数 噪声值  动量值      表示使用科学系的变换参数(lambda, beta)
    bn2d = nn.BatchNorm2d(num_features=2, eps=1e-5, momentum=0.1, affine=True)
    # 3.对数据进行批量归一化处理
    output_2d = bn2d(input_2d)
    print(f'二维数据处理后为:{output_2d}')


In [22]:
def BN_1n():
    input_1d = torch.randn(size=(2,2))
    linear1 = nn.Linear(2, 4)
    l1 = linear1(input_1d)
    bn1d = nn.BatchNorm1d(num_features=4)
    output_1d = bn1d(l1)
    print(f'一维数据处理后为:{output_1d}')


In [23]:
if __name__ == '__main__':
    # dropout()
    BN_1n()
    BN_2n()

处理后的数据为:tensor([[-1.0000, -0.9996, -1.0000, -1.0000],
        [ 1.0000,  0.9996,  1.0000,  1.0000]],
       grad_fn=<NativeBatchNormBackward0>)
处理后的数据为:tensor([[[[ 1.2114, -0.4714,  1.0739,  0.5832],
          [-1.1257, -0.9357,  1.2342, -0.1843],
          [-0.2008, -1.5649, -0.9404,  1.3203]],

         [[-1.8776, -1.1643,  0.6960,  0.6039],
          [-0.5598,  0.9532,  1.1675,  0.3250],
          [-0.4375, -1.0426,  1.5071, -0.1710]]]],
       grad_fn=<NativeBatchNormBackward0>)
